# Download Droid

In [40]:
import gcsfs
from pathlib import Path
import shutil
import json
from typing import Any
import sys
import os

In [41]:
gcsfs_fs = gcsfs.GCSFileSystem(token="anon")
GCLOUD_DIR = Path("gresearch/robotics/droid_raw/1.0.1")
OUTPUT_DIR_RAW = Path("../dataset/raw")
OURPUT_DIR_RRD = Path("../dataset/rrd")
MIN_FREE_BYTES = 10 * 1024**3

EPISODE_FROM, EPISODE_TO = 0, 100

## Download manifest

In [42]:
def load_manifest(
        fs: gcsfs.GCSFileSystem, 
        gcloud_path: Path, 
        output_dir: Path, 
        refresh: bool
    ) -> list[str]:
    """Return the sorted GCS paths of all successful-episode metadata JSONs, cached locally."""
    manifest = output_dir / "manifest.txt"
    os.makedirs(output_dir, exist_ok=True)
    if manifest.exists() and not refresh:
        paths = [line.strip() for line in manifest.read_text().splitlines() if line.strip()]
    else:
        print("Listing successful raw DROID episodes on GCS (takes a few minutes)...", flush=True)
        paths = sorted(fs.glob(f"{gcloud_path}/*/success/*/*/metadata_*.json"))
        if not paths:
            raise RuntimeError("No DROID raw episodes found")
        temporary = manifest.with_suffix(".partial")
        temporary.write_text("\n".join(paths) + "\n")
        temporary.replace(manifest)

    return paths

In [43]:
def download_episode(fs, episode_dir: str, destination: Path, overwrite: bool) -> dict[str, Path]:
    """Download every file under episode_dir, preserving its relative layout."""
    destination = Path(destination)
    local = {}
    for remote, info in fs.find(episode_dir, detail=True).items(): 
        relative = remote.removeprefix(episode_dir + "/")
        path = destination / relative
        if overwrite or not path.exists() or path.stat().st_size != info["size"]:
            path.parent.mkdir(parents=True, exist_ok=True)
            fs.get(remote, str(path))
        local[relative] = path
    return local

In [44]:
def process_episode(
    fs: gcsfs.GCSFileSystem,
    metadata_path: str,
    index: int,
    output_dir: Path,
    overwrite: bool,
) -> bool:
    """Download, convert, and atomically write one episode. Returns True on success."""

    with fs.open(metadata_path, "rt") as file:
        metadata = json.load(file)

    final_path = output_dir / f"episode-{index:06d}-{metadata['uuid']}.rrd"

    if final_path.exists() and not overwrite:
        print(f"[{index:06d}] already converted; skipping")
        return True

    episode_dir = metadata_path.rsplit("/", 1)[0]

    download_episode(fs, episode_dir, output_dir / metadata["uuid"])

    return download_episode


In [45]:
paths = load_manifest(gcsfs_fs, GCLOUD_DIR, OUTPUT_DIR_RAW, False)

episodes = []   # (index, metadata, local_files) shared by the next two loops

for index, metadata_path in enumerate(paths[EPISODE_FROM:EPISODE_TO], start=EPISODE_FROM):
    if shutil.disk_usage(OUTPUT_DIR_RAW).free < MIN_FREE_BYTES:
        print(f"Stopping: less than {MIN_FREE_BYTES / 1024**3:.0f} GiB free", file=sys.stderr)
        break
    try:
        with gcsfs_fs.open(metadata_path, "rt") as file:
            episode_metadata = json.load(file)
        episode_dir = metadata_path.rsplit("/", 1)[0]
        local_files = download_episode(
            gcsfs_fs, episode_dir, OUTPUT_DIR_RAW / episode_metadata["uuid"], False
        )
        episodes.append((index, episode_metadata, local_files))
        print(f"[{index:06d}] downloaded {episode_metadata['uuid']}")
    except Exception as error:  # noqa: BLE001 - one bad episode must not end the run
        print(f"[{index:06d}] FAILED: {error}", file=sys.stderr)

[000000] downloaded AUTOLab+5d05c5aa+2023-07-07-09h-42m-23s
[000001] downloaded AUTOLab+5d05c5aa+2023-07-07-09h-43m-39s
[000002] downloaded AUTOLab+5d05c5aa+2023-07-07-09h-44m-34s
[000003] downloaded AUTOLab+5d05c5aa+2023-07-07-09h-52m-29s
[000004] downloaded AUTOLab+5d05c5aa+2023-07-07-09h-53m-50s
[000005] downloaded AUTOLab+5d05c5aa+2023-07-07-09h-55m-14s
[000006] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-01m-42s
[000007] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-03m-36s
[000008] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-04m-33s
[000009] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-05m-20s
[000010] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-06m-23s
[000011] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-07m-45s
[000012] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-09m-06s
[000013] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-11m-29s
[000014] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-12m-24s
[000015] downloaded AUTOLab+5d05c5aa+2023-07-07-10h-13m-22s
[000016] downloaded AUTOLab+5d05c5aa+202

# Download URDF

In [46]:
import io
import tarfile
import urllib.request

URL = "https://github.com/rerun-io/python-example-droid-dataset/archive/refs/heads/master.tar.gz"
DEST = Path("../dataset/assets")

payload = urllib.request.urlopen(URL).read()
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as tar:
    members = [m for m in tar.getmembers() if "_description/" in m.name]
    for m in members:
        m.name = m.name.split("/", 1)[1]
    tar.extractall(DEST, members=members, filter="data")

URDF = DEST / "panda.urdf"
shutil.copyfile(DEST / "franka_description/panda.urdf", URDF)
print(URDF)

../dataset/assets/panda.urdf


# Convert data to rerun

In [47]:
import subprocess

def transcode_videos(local_files, destination):
    """Re-encode every mp4 without B-frames; returns local_files with the new paths."""
    destination.mkdir(parents=True, exist_ok=True)
    result = dict(local_files)
    for relative in [r for r in local_files if r.endswith(".mp4")]:
        out = destination / Path(relative).name
        if not out.exists():                       # cached: rerunning the cell is free
            subprocess.run([
                "ffmpeg", "-y", "-v", "error", "-i", str(local_files[relative]),
                "-c:v", "libx264", "-bf", "0", "-g", "30", "-crf", "18",
                "-preset", "veryfast", "-fps_mode", "passthrough", "-an", str(out),
            ], check=True)
        result[relative] = out
    return result

def mp4_frames(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=nb_frames", "-of", "csv=p=0", str(path)],
        capture_output=True, text=True, check=True)
    return int(out.stdout)


In [48]:
import re
import h5py
import numpy as np

CAMERAS = ("wrist", "ext1", "ext2")

def robot_timestamps_ns(trajectory: h5py.File) -> np.ndarray:
    stamps = trajectory["observation/timestamp/robot_state"]
    seconds = np.asarray(stamps["robot_timestamp_seconds"], dtype=np.int64)
    nanos = np.asarray(stamps["robot_timestamp_nanos"], dtype=np.int64)
    return seconds * 1_000_000_000 + nanos

def validate_episode(local_files, metadata, cameras=CAMERAS):
    uuid = metadata["uuid"]
    if not re.fullmatch(r"[A-Za-z0-9._+-]+", uuid):
        raise RuntimeError(f"unsafe uuid for filenames: {uuid!r}")

    with h5py.File(local_files["trajectory.h5"], "r") as trajectory:
        rows = int(metadata["trajectory_length"])
        stamps = robot_timestamps_ns(trajectory)

        if len(stamps) != rows:
            raise RuntimeError(f"metadata says {rows} steps, h5 has {len(stamps)}")
        if not np.all(np.diff(stamps) >= 0):
            raise RuntimeError("robot timestamps out of order")

        wrong = []
        trajectory.visititems(
            lambda name, obj: wrong.append(name)
            if isinstance(obj, h5py.Dataset) and len(obj) != rows
            else None
        )
        if wrong:
            raise RuntimeError(f"expected {rows} rows in every dataset, not in: {wrong}")

        for camera in cameras:
            serial = metadata[f"{camera}_cam_serial"]
            for path in (
                f"observation/timestamp/cameras/{serial}_estimated_capture",
                f"observation/camera_extrinsics/{serial}_left",
            ):
                if path not in trajectory:
                    raise RuntimeError(f"missing {path}")

In [49]:
for position, (index, episode_metadata, local_files) in enumerate(episodes):
    local_files = transcode_videos(
        local_files, OUTPUT_DIR_RAW / episode_metadata["uuid"] / "no-b-frames"
    )
    validate_episode(local_files, episode_metadata)
    episodes[position] = (index, episode_metadata, local_files)   # keep transcoded paths
    print(f"[{index:06d}] transcoded and validated")

[000000] transcoded and validated
[000001] transcoded and validated
[000002] transcoded and validated
[000003] transcoded and validated
[000004] transcoded and validated
[000005] transcoded and validated
[000006] transcoded and validated
[000007] transcoded and validated
[000008] transcoded and validated
[000009] transcoded and validated
[000010] transcoded and validated
[000011] transcoded and validated
[000012] transcoded and validated
[000013] transcoded and validated
[000014] transcoded and validated
[000015] transcoded and validated
[000016] transcoded and validated
[000017] transcoded and validated
[000018] transcoded and validated
[000019] transcoded and validated
[000020] transcoded and validated
[000021] transcoded and validated
[000022] transcoded and validated
[000023] transcoded and validated
[000024] transcoded and validated
[000025] transcoded and validated
[000026] transcoded and validated
[000027] transcoded and validated
[000028] transcoded and validated
[000029] trans

## Setup

In [50]:
import h5py
import numpy as np
import pyarrow as pa
import rerun as rr
import rerun.blueprint as rrb
from rerun.urdf import UrdfTree
from scipy.spatial.transform import Rotation, RigidTransform

APPLICATION_ID = "droid"
TIMELINE = "real_time"
CAMERAS = ("wrist", "ext1", "ext2")
ARM_JOINTS = [f"panda_joint{i + 1}" for i in range(7)]

# Rectified left-camera intrinsics per serial, read once from the .svo files with the ZED
# SDK (the MP4 exports are rectified, so the raw factory calibration is a few pixels off).
# Wrist is a ZED-M, the external cameras are ZED 2s.
CAMERA_RESOLUTION = (1280, 720)
CAMERA_INTRINSICS = {
    "18026681": np.array([[732.202, 0.0, 640.044], [0.0, 732.202, 365.788], [0.0, 0.0, 1.0]]),
    "22008760": np.array([[524.546, 0.0, 639.777], [0.0, 524.546, 370.278], [0.0, 0.0, 1.0]]),
    "24400334": np.array([[531.915, 0.0, 636.152], [0.0, 531.915, 344.008], [0.0, 0.0, 1.0]]),
}

# Where the gripper's tool centre point sits relative to the flange, [x, y, z, rx, ry, rz].
# DROID's cartesian_position is the flange, so everything tool-related hangs off this.
TCP_FROM_FLANGE = np.array([0.0, 0.0, 0.15, 0.0, 0.0, 0.0])

# A gripper "event" is a change of more than GRASP_THRESHOLD over GRASP_SPAN steps.
GRASP_THRESHOLD = 0.4
GRASP_SPAN = 5

# Overlay styling.
TARGET_COLOR = [255, 60, 60]
PATH_COLOR = [160, 80, 255]
CROSS_RADIUS = 14.0

In [51]:
urdf = UrdfTree.from_file_path(URDF, "robot")

In [52]:
def robot_timestamps_ns(trajectory):
    """The control loop's own clock, in nanoseconds since the epoch."""
    stamps = trajectory["observation/timestamp/robot_state"]
    seconds = np.asarray(stamps["robot_timestamp_seconds"], dtype=np.int64)
    nanos = np.asarray(stamps["robot_timestamp_nanos"], dtype=np.int64)
    return seconds * 1_000_000_000 + nanos


def to_transform(v):
    """[x, y, z, rx, ry, rz] -> SE(3). Works for one pose (6,) or a batch (N, 6).

    DROID stores rotations as extrinsic x-y-z Euler angles, which is what
    `Rotation.from_euler("xyz", ...)` decodes (lowercase = extrinsic).
    """
    v = np.asarray(v, dtype=float)
    return RigidTransform.from_components(v[..., :3], Rotation.from_euler("xyz", v[..., 3:]))


def to_pixels(points_cam, K, resolution):
    """(N, 3) camera-frame points -> (uv, ok); `ok` masks behind-camera and off-frame."""
    uv = points_cam @ K.T
    uv = uv[:, :2] / uv[:, 2:3]
    ok = ((points_cam[:, 2] > 0)
          & (uv[:, 0] >= 0) & (uv[:, 0] < resolution[0])
          & (uv[:, 1] >= 0) & (uv[:, 1] < resolution[1]))
    return uv, ok


def gripper_events(values, threshold=GRASP_THRESHOLD, span=GRASP_SPAN):
    """Indices where the gripper moved more than `threshold` over `span` steps.

    A wide span catches slow squeezes a neighbour-diff misses, but reports the step where
    the motion *finished*; only the first index of each run is kept.
    """
    changed = np.abs(values[span:] - values[:-span]) > threshold
    ids = np.arange(span, len(values))[changed]
    return ids[np.concatenate([[True], np.diff(ids) > 1])] if len(ids) else ids


def video_stream(path, entity_path, capture_ns, timeline=TIMELINE):
    """Read an mp4 as a VideoStream, restamped from mp4 PTS to real capture times.

    The reader stamps every sample with the mp4's own clock, which claims 60 fps while the
    cameras actually ran at ~14 Hz -- so the time column is rewritten chunk by chunk. The
    file must be B-frame free (see `transcode_videos`).
    """
    cursor = 0

    def retag(chunk):
        nonlocal cursor
        if chunk.is_static:                          # codec chunk: no timeline to fix
            return chunk
        batch = chunk.to_record_batch()
        column = batch.schema.get_field_index(timeline)
        field = batch.schema.field(column)
        times = pa.array(capture_ns[cursor:cursor + chunk.num_rows].astype("datetime64[ns]"))
        cursor += chunk.num_rows
        # metadata= is load-bearing: it is what marks the column as a timeline.
        new_field = pa.field(field.name, times.type, nullable=field.nullable,
                             metadata=field.metadata)
        return rr.experimental.Chunk.from_record_batch(
            batch.set_column(column, new_field, times))[0]

    reader = rr.experimental.Mp4Reader(path, entity_path=entity_path,
                                       timeline_name=timeline, timeline_type="timestamp")
    return reader.stream().map(retag)

## Logging

In [53]:
def log_robot_state(rec, trajectory, indexes, urdf_root, tcp_from_flange=TCP_FROM_FLANGE):
    """Per-step signals, plus the flange and the tool point as 3D frames."""
    state = trajectory["observation/robot_state"]
    for name in ("gripper_position", "joint_positions", "joint_velocities"):
        rec.send_columns(
            f"/robot_state/{name}", indexes=indexes,
            columns=rr.Scalars.columns(scalars=state[name][:])
        )

    pose = state["cartesian_position"][:]
    rec.send_columns(
        "/scene/end_effector",
        indexes=indexes,
        columns=rr.Transform3D.columns(
            translation=pose[:, :3],
            quaternion=Rotation.from_euler("xyz", pose[:, 3:]).as_quat(),
            parent_frame=[urdf_root] * len(pose),
            child_frame=["end_effector"] * len(pose),
        ),
    )
    # One static row: the tool offset rides along with the flange for the whole episode.
    rec.log("/scene/end_effector/tcp",
        rr.Transform3D(
            translation=tcp_from_flange[:3],
            parent_frame="end_effector", 
            child_frame="tcp"
        ), 
        static=True
    )
    rec.log("/scene/end_effector/tcp", rr.CoordinateFrame("tcp"), static=True)
    rec.log("/scene/end_effector/tcp", rr.TransformAxes3D(0.1), static=True)


def log_robot_model(rec, trajectory, times_ns, urdf, timeline=TIMELINE, arm_joints=ARM_JOINTS):
    """The URDF meshes, plus one Transform3D per joint per step from forward kinematics."""
    rr.experimental.send_chunks(
        urdf.stream().drop(content=f"/robot/{urdf.name}/collision_geometries/**"),
        recording=rec
    )

    arm = trajectory["observation/robot_state/joint_positions"][:]
    batches = urdf.compute_joint_transform_batches(
        pa.array([arm_joints] * len(arm), type=pa.list_(pa.string())),
        pa.array(arm.tolist(), type=pa.list_(pa.float64())),
    )
    entries = batches.flatten()
    per_step = len(batches[0])          # one transform per joint we drove

    rec.send_columns(
        "/robot/transforms",
        indexes=[rr.TimeColumn(timeline, timestamp=np.repeat(times_ns, per_step))],
        columns=rr.Transform3D.columns(
            translation=np.asarray(entries.field("translation").flatten()).reshape(-1, 3),
            quaternion=np.asarray(entries.field("quaternion").flatten()).reshape(-1, 4),
            parent_frame=entries.field("parent_frame").to_pylist(),
            child_frame=entries.field("child_frame").to_pylist(),
        ),
    )


def log_cameras(
    rec, trajectory, metadata, local_files, indexes, urdf_root,
    cameras=CAMERAS, timeline=TIMELINE,
    intrinsics=CAMERA_INTRINSICS, resolution=CAMERA_RESOLUTION
):
    """Video, pose and intrinsics per camera.

    Every entity needs a CoordinateFrame: Rerun places entities by frame, not by path, and
    an unnamed entity gets an implicit frame with no edge to the robot.
    """
    for name in cameras:
        serial = metadata[f"{name}_cam_serial"]
        entity = f"/cameras/{name}"

        capture_ns = trajectory[f"observation/timestamp/cameras/{serial}_estimated_capture"][:]
        rr.experimental.send_chunks(
            video_stream(
                local_files[f"recordings/MP4/{serial}.mp4"], 
                entity,
                capture_ns * 1_000_000,
                timeline
            ),
            recording=rec
        )

        extrinsics = trajectory[f"observation/camera_extrinsics/{serial}_left"][:]
        rec.send_columns(
            entity,
            indexes=indexes,            # robot clock: extrinsics are written per control step
            columns=rr.Transform3D.columns(
                translation=extrinsics[:, :3],
                quaternion=Rotation.from_euler("xyz", extrinsics[:, 3:]).as_quat(),
                parent_frame=[urdf_root] * len(extrinsics),
                child_frame=[f"cam_{name}"] * len(extrinsics),
            ),
        )
        rec.log(entity, rr.CoordinateFrame(f"cam_{name}"), static=True)
        # child_frame must match the transform above: the camera visualizer reads
        # Pinhole.child_frame, not CoordinateFrame.
        rec.log(
            entity, 
            rr.Pinhole(
                image_from_camera=intrinsics[serial],
                resolution=resolution,
                child_frame=f"cam_{name}"
            ), 
            static=True
        )


def log_grasp_targets(
    rec, trajectory, metadata, times_ns,
    cameras=CAMERAS, timeline=TIMELINE,
    intrinsics=CAMERA_INTRINSICS, resolution=CAMERA_RESOLUTION,
    tcp_from_flange=TCP_FROM_FLANGE, target_color=TARGET_COLOR,
    path_color=PATH_COLOR, cross_radius=CROSS_RADIUS
):
    """Where the tool point is heading for the next gripper event, drawn on each camera.

    Overlays are child entities of the camera, so they land in image space, stay toggleable
    in the entity tree, and never touch the pixels. Anything behind the camera or outside
    the frame is dropped before logging.
    """
    events = gripper_events(trajectory["action/gripper_position"][:])
    if len(events) == 0:
        return
    tcp_world = to_transform(trajectory["action/cartesian_position"][:]) \
        * to_transform(tcp_from_flange)

    steps = np.arange(len(times_ns))
    slot = np.searchsorted(events, steps, "right")
    has_goal = slot < len(events)
    goal = events[np.clip(slot, 0, len(events) - 1)]        # next gripper event per step

    for name in cameras:
        serial = metadata[f"{name}_cam_serial"]
        cams = to_transform(trajectory[f"observation/camera_extrinsics/{serial}_left"][:])
        K = intrinsics[serial]

        # Target: fully vectorised -- batch of camera inverses x batch of goal poses.
        uv, ok = to_pixels((cams.inv() * tcp_world[goal]).translation, K, resolution)
        keep = ok & has_goal
        arms = [
            arm for x, y in uv[keep] for arm in (
                np.array([[x - cross_radius, y], [x + cross_radius, y]]),
                np.array([[x, y - cross_radius], [x, y + cross_radius]]),
            )
        ]
        rec.send_columns(
            f"/cameras/{name}/target",
            indexes=[rr.TimeColumn(timeline, timestamp=times_ns[keep])],
            columns=rr.LineStrips2D.columns(strips=arms, colors=[target_color] * len(arms))
                      .partition(np.full(keep.sum(), 2)),   # two strips make one cross
        )

        # Path: each row is a different length, so build the list and send it in one call.
        strips, when = [], []
        for step in steps[has_goal]:
            points, good = to_pixels(
                (cams[step].inv() * tcp_world[step:goal[step] + 1]).translation, K, resolution
            )
            if good.sum() > 1:
                strips.append(points[good])
                when.append(times_ns[step])
        rec.send_columns(
            f"/cameras/{name}/future_path",
            indexes=[rr.TimeColumn(timeline, timestamp=np.array(when))],
            columns=rr.LineStrips2D.columns(strips=strips, colors=[path_color] * len(strips)),
        )

        for overlay in ("target", "future_path"):
            rec.log(f"/cameras/{name}/{overlay}", rr.CoordinateFrame(f"cam_{name}"), static=True)

In [54]:
def make_blueprint(cameras=CAMERAS):
    """Robot and cameras in 3D on the left, one window per camera and the gripper right."""
    return rrb.Blueprint(
        rrb.Horizontal(
            rrb.Spatial3DView(name="Scene", origin="/"),
            rrb.Vertical(
                rrb.Horizontal(
                    *(
                        rrb.Spatial2DView(
                            origin=f"/cameras/{name}", name=name) 
                            for name in cameras
                    )
                ),
                rrb.TimeSeriesView(
                    name="Gripper", 
                    origin="/robot_state/gripper_position"
                ),
                row_shares=[3, 1],
            ),
            column_shares=[1, 1],
        ),
        collapse_panels=True,
    )


def convert_episode(
    metadata, local_files, urdf, output_path=None,
    application_id=APPLICATION_ID, timeline=TIMELINE
):
    """Build one recording. Saves to `output_path` when given; returns the stream."""
    # The id doubles as the server's segment id, which travels in URL query strings where
    # "+" decodes to a space -- so the "+" in DROID uuids must go.
    episode_id = metadata["uuid"].replace("+", "_")
    rec = rr.RecordingStream(application_id, recording_id=episode_id)
    if output_path is not None:
        rec.save(output_path, default_blueprint=make_blueprint())

    # The episode's scalar metadata rides along in the recording's own info panel.
    rec.send_recording_name(episode_id)
    rec.send_property("episode",
        rr.AnyValues(**{
            key: value for key, value in metadata.items() if isinstance(value, (str, int, float))
        })
    )

    urdf_root = urdf.root_link().name   # "panda_link0": the frame every pose hangs off

    with h5py.File(local_files["trajectory.h5"], "r") as trajectory:
        times_ns = robot_timestamps_ns(trajectory).astype("datetime64[ns]")
        indexes = [rr.TimeColumn(timeline, timestamp=times_ns)]

        log_robot_state(rec, trajectory, indexes, urdf_root)
        log_robot_model(rec, trajectory, times_ns, urdf, timeline=timeline)
        log_cameras(rec, trajectory, metadata, local_files, indexes, urdf_root, timeline=timeline)
        log_grasp_targets(rec, trajectory, metadata, times_ns, timeline=timeline)

    rec.flush()
    return rec

## Convert every downloaded episode

In [57]:
OURPUT_DIR_RRD.mkdir(parents=True, exist_ok=True)

for index, episode_metadata, local_files in episodes:
    output_path = OURPUT_DIR_RRD / f"episode-{index:06d}-{episode_metadata['uuid']}.rrd"
    convert_episode(episode_metadata, local_files, urdf, output_path)
    print(f"[{index:06d}] wrote {output_path.name} "
          f"({output_path.stat().st_size / 1024**2:.0f} MiB)")

[000000] wrote episode-000000-AUTOLab+5d05c5aa+2023-07-07-09h-42m-23s.rrd (25 MiB)
[000001] wrote episode-000001-AUTOLab+5d05c5aa+2023-07-07-09h-43m-39s.rrd (26 MiB)
[000002] wrote episode-000002-AUTOLab+5d05c5aa+2023-07-07-09h-44m-34s.rrd (29 MiB)
[000003] wrote episode-000003-AUTOLab+5d05c5aa+2023-07-07-09h-52m-29s.rrd (32 MiB)
[000004] wrote episode-000004-AUTOLab+5d05c5aa+2023-07-07-09h-53m-50s.rrd (26 MiB)
[000005] wrote episode-000005-AUTOLab+5d05c5aa+2023-07-07-09h-55m-14s.rrd (33 MiB)
[000006] wrote episode-000006-AUTOLab+5d05c5aa+2023-07-07-10h-01m-42s.rrd (28 MiB)
[000007] wrote episode-000007-AUTOLab+5d05c5aa+2023-07-07-10h-03m-36s.rrd (25 MiB)
[000008] wrote episode-000008-AUTOLab+5d05c5aa+2023-07-07-10h-04m-33s.rrd (23 MiB)
[000009] wrote episode-000009-AUTOLab+5d05c5aa+2023-07-07-10h-05m-20s.rrd (29 MiB)
[000010] wrote episode-000010-AUTOLab+5d05c5aa+2023-07-07-10h-06m-23s.rrd (33 MiB)
[000011] wrote episode-000011-AUTOLab+5d05c5aa+2023-07-07-10h-07m-45s.rrd (24 MiB)
[000

## Preview one episode

In [58]:
_, episode_metadata, local_files = episodes[0]
rec = recording=convert_episode(episode_metadata, local_files, urdf)
rr.notebook_show(recording=rec)

HTML(value='<div id="46387fad-4463-48f0-b712-9615601f7fd4"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…